Aim of this notebook is to identify what is going on with the bias & residual standard deviation of the empirical GMM used in this study?  
Is this simply a data issue or is there some other issue?

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

import sim_ranking as sr
import ml_tools as mlt

In [ ]:
nzgmdb_ffp = Path("/Users/claudy/dev/work/data/gm_datasets/nz_gmdb/v4.2/custom/mod_ground_motion_im_table_rotd50_flat.csv")
# gnn_results_dir = Path("/Users/claudy/dev/work/data/sim_ranking/results/gnn/0306_1947_cv_v4p2NZGMDB_v2p3_4e8s_fixedZ1p0Issue")
gnn_results_dir = Path("/Users/claudy/dev/work/data/sim_ranking/results/gnn/0317_1033_cv_v4p2NZGMDB_v2p4_4e8s_MagDistFilterFix")
emp_gm_params_ffp = Path("/Users/claudy/dev/work/data/sim_ranking/emp_gm_params/nzgmdb_v4p2/emp_gm_params_400MaxRjb.parquet")

In [ ]:
obs_data = sr.data.load_obs_nzgmdb(nzgmdb_ffp)
print(f"Number of records: {obs_data.n_records}")

In [ ]:
gnn_val_results = pd.read_parquet(gnn_results_dir / "val_results.parquet")

In [ ]:
emp_gm_params = pd.read_parquet(emp_gm_params_ffp)
assert gnn_val_results.index.isin(emp_gm_params.index).all()

In [ ]:
ims = sr.constants.PSA_KEYS

In [ ]:
# Marginal results
mar_results = emp_gm_params.copy(deep=True)
mar_results = mar_results.rename(columns={cur_col: cur_col.replace("_mean", "_pred") for cur_col in mar_results.columns if cur_col.endswith("_mean")})
mar_results = mar_results.rename(columns={cur_col: cur_col.replace("_std_Total", "_pred_std") for cur_col in mar_results.columns if cur_col.endswith("_std_Total")})
mar_results = mar_results.drop(columns=[cur_col for cur_col in mar_results.columns if cur_col.endswith("_std_Inter") or cur_col.endswith("_std_Intra")])
mar_results = mar_results.loc[gnn_val_results.index]
mar_results[sr.constants.PSA_KEYS] = gnn_val_results[sr.constants.PSA_KEYS]
mar_results["site_int"] = gnn_val_results["site_int"]

In [ ]:
# Compute marginal residuals
mar_res_df = sr.ml.gnn_gm.get_residuals(mar_results, ims=sr.constants.PSA_KEYS)

mar_res_mean_std_df = pd.concat((mar_res_df[sr.constants.PSA_KEYS].mean(axis=0), mar_res_df[sr.constants.PSA_KEYS].std(axis=0)), axis=1)
mar_res_mean_std_df.columns = ["mean", "std"]

In [ ]:
# Compute Residuals for tectonic types
crustal_keys = mar_results.loc[obs_data.record_df.loc[mar_results.index, "tect_type"] == "crustal"].index
crustal_bias_std_df = pd.concat((mar_res_df.loc[crustal_keys, sr.constants.PSA_KEYS].mean(axis=0), mar_res_df.loc[crustal_keys, sr.constants.PSA_KEYS].std(axis=0)), axis=1)
crustal_bias_std_df.columns = ["mean", "std"]

subduction_keys = mar_results.loc[obs_data.record_df.loc[mar_results.index, "tect_type"].isin(["subduction_interface", "subduction_slab"])].index
subduction_bias_std_df = pd.concat((mar_res_df.loc[subduction_keys, sr.constants.PSA_KEYS].mean(axis=0), mar_res_df.loc[subduction_keys, sr.constants.PSA_KEYS].std(axis=0)), axis=1)
subduction_bias_std_df.columns = ["mean", "std"]

other_keys = mar_results.loc[~mar_results.index.isin(crustal_keys) & ~mar_results.index.isin(subduction_keys)].index
other_bias_std_df = pd.concat((mar_res_df.loc[other_keys, sr.constants.PSA_KEYS].mean(axis=0), mar_res_df.loc[other_keys, sr.constants.PSA_KEYS].std(axis=0)), axis=1)
other_bias_std_df.columns = ["mean", "std"]

print("Crustal keys: ", len(crustal_keys))
print("Subduction keys: ", len(subduction_keys))
print("Other keys: ", len(other_keys))

## Bias & Residual Standard Deviation

In [ ]:
fig, (ax1, ax2)  = plt.subplots(1, 2, figsize=(16, 6))

ax1.semilogx(sr.constants.PERIODS, mar_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"], label="All", c="black", linewidth=2.0)
ax1.semilogx(sr.constants.PERIODS, crustal_bias_std_df.loc[sr.constants.PSA_KEYS, "mean"], label="Crustal", c="red")
ax1.semilogx(sr.constants.PERIODS, subduction_bias_std_df.loc[sr.constants.PSA_KEYS, "mean"], label="Subduction", c="green")
ax1.semilogx(sr.constants.PERIODS, other_bias_std_df.loc[sr.constants.PSA_KEYS, "mean"], label="Other", c="orange")

ax1.set_xlim(0.01, 10)
ax1.set_ylim(-1.5, 1.5)
ax1.grid(which="both", linewidth=0.5, alpha=0.5, linestyle="--")
ax1.axhline(0, color="black")
ax1.legend()


ax2.semilogx(sr.constants.PERIODS, mar_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], label="GMM", c="black", linewidth=2.0)
ax2.semilogx(sr.constants.PERIODS, crustal_bias_std_df.loc[sr.constants.PSA_KEYS, "std"], label="Crustal", c="red")
ax2.semilogx(sr.constants.PERIODS, subduction_bias_std_df.loc[sr.constants.PSA_KEYS, "std"], label="Subduction", c="green")
ax2.semilogx(sr.constants.PERIODS, other_bias_std_df.loc[sr.constants.PSA_KEYS, "std"], label="Other", c="orange")

ax2.set_xlim(0.01, 10)
ax2.set_ylim(0, 1.5)
ax2.grid(which="both", linewidth=0.5, alpha=0.5, linestyle="--")

fig.tight_layout()

## Trends - Magnitude

In [ ]:
mar_res_df["mag"] = obs_data.event_df.loc[mar_res_df.event_id, "mag"].values
mar_results["mag"] = obs_data.event_df.loc[mar_results.event_id, "mag"].values

In [ ]:
# Magnitude bins
mag_bins = [3, 4, 4.5, 5, 6, 8]
assert mar_res_df.mag.min() >= mag_bins[0] and mar_res_df.mag.max() <= mag_bins[-1]
mag_bin_keys = [f"{mag_bins[i]}_{mag_bins[i+1]}" for i in range(len(mag_bins) - 1)]
mar_res_df["mag_bin"] = pd.cut(mar_res_df.mag, bins=mag_bins, labels=mag_bin_keys)
mar_results["mag_bin"] = pd.cut(mar_results.mag, bins=mag_bins, labels=mag_bin_keys)
res_mag_groups = mar_res_df.groupby("mag_bin")

### Magnitude Distribution

In [ ]:
# Distribution
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
sns.histplot(mar_res_df.mag, bins=25, ax=ax)
ax.set_title("Magnitude Distribution")
ax.set_xlabel("Magnitude")

for i, (cur_bin, cur_key) in enumerate(zip(mag_bins, mag_bin_keys)):
	ax.axvline(cur_bin, color="k", linestyle="--", linewidth=1.0)
	ax.text((mag_bins[i] + mag_bins[i+1]) / 2, ax.get_ylim()[1] * 0.98, f"N={res_mag_groups.size()[cur_key]}", horizontalalignment="center", verticalalignment="top")
   
ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax.set_xlim(3, 8)
fig.tight_layout()

### Magnitude Bias & (Residual) Standard Deviation

In [ ]:
res_mag_bias = res_mag_groups[ims].mean()
res_mag_bias_std = res_mag_groups[ims].std()

colors = list(reversed(cm.viridis(np.linspace(0, 1, len(mag_bin_keys))))) 

fig, (ax1, ax2)  = plt.subplots(1, 2, figsize=(16, 6))

ax1.set_xlim(0.01, 10)
ax1.set_ylim(-1.5, 1.5)
ax1.grid(which="both", linewidth=0.5, alpha=0.5, linestyle="--")
ax1.axhline(0, color="black")

ax2.set_xlim(0.01, 10)
ax2.set_ylim(0, 1.5)
ax2.grid(which="both", linewidth=0.5, alpha=0.5, linestyle="--")

# Bias
for i, (cur_key, cur_group) in enumerate(res_mag_groups):
	ax1.semilogx(sr.constants.PERIODS, res_mag_bias.loc[cur_key, sr.constants.PSA_KEYS], label=f"{cur_key} - N = {res_mag_groups.size()[cur_key]}", c=colors[i])
ax1.semilogx(sr.constants.PERIODS, mar_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"], c="b", label="Overall")
ax1.legend()

# Residual Standard Deviation
for i, (cur_key, cur_group) in enumerate(res_mag_groups):
	ax2.semilogx(sr.constants.PERIODS, res_mag_bias_std.loc[cur_key, sr.constants.PSA_KEYS], label=cur_key, c=colors[i])
    
ax2.semilogx(sr.constants.PERIODS, mar_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], c="b")

fig.tight_layout()

In [ ]:
plot_ims = ["pSA_0.01", "pSA_0.05", "pSA_0.1", "pSA_0.5", "pSA_1.0", "pSA_3.0", "pSA_5.0", "pSA_10.0"]
n_points_per_bin = 1000

fig, axs = mlt.plotting.get_fig_axes(8, 2, -1, (8, 6))

for cur_ax, cur_im in zip(axs, plot_ims):
    cur_ax.scatter(mar_res_df.mag, mar_res_df[cur_im], alpha=0.5, s=5)

    bin_centres, bin_means, bin_stds = mlt.utils.compute_count_binned_trend(mar_res_df.mag.values, mar_res_df[cur_im].values, n_points_per_bin, ignore_nans=True, verbose=False)
    cur_n_points = np.count_nonzero(~np.isnan(mar_res_df[cur_im]))
    cur_ax.plot(bin_centres, bin_means, c="r")
    cur_ax.plot(bin_centres, bin_means - bin_stds, linestyle="--", color="r", linewidth=2)
    cur_ax.plot(bin_centres, bin_means + bin_stds, linestyle="--", color="r", linewidth=2)

    cur_ax.set_title(f"{sr.utils.get_nice_im_name(cur_im)} - N={cur_n_points}")
    cur_ax.set_xlabel("Magnitude")
    cur_ax.set_ylabel("Residual")
    cur_ax.set_ylim(-4, 4)
    cur_ax.grid(which="both", linewidth=0.5, alpha=0.5, linestyle="--")
    cur_ax.axhline(0, color="black")

fig.tight_layout()

## Trends - $R_{Rup}$

In [ ]:
mar_res_df["rrup"] = obs_data.record_df.loc[mar_results.index, "rrup"].values
mar_results["rrup"] = obs_data.record_df.loc[mar_results.index, "rrup"].values

In [ ]:
# Distance bins
dist_bins = [0, 25, 50, 100, 150, 225]
assert mar_res_df.rrup.min() >= dist_bins[0] and mar_res_df.rrup.max() <= dist_bins[-1]
dist_bin_keys = [f"{dist_bins[i]}_{dist_bins[i+1]}" for i in range(len(dist_bins) - 1)]
mar_res_df["dist_bin"] = pd.cut(mar_res_df.rrup, bins=dist_bins, labels=dist_bin_keys)
mar_results["dist_bin"] = pd.cut(mar_results.rrup, bins=dist_bins, labels=dist_bin_keys)
res_dist_groups = mar_res_df.groupby("dist_bin")

### $R_{Rup}$ Distribution

In [ ]:
# Distribution
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
sns.histplot(mar_res_df.rrup, bins=25, ax=ax)
ax.set_title("$R_{Rup}$ Distribution")
ax.set_xlabel("$R_{Rup}$")

for i, (cur_bin, cur_key) in enumerate(zip(dist_bins, dist_bin_keys)):
	ax.axvline(cur_bin, color="k", linestyle="--", linewidth=1.0)
	ax.text((dist_bins[i] + dist_bins[i+1]) / 2, ax.get_ylim()[1] * 0.98, f"N={res_dist_groups.size()[cur_key]}", horizontalalignment="center", verticalalignment="top")
    
ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax.set_xlim(0, 200)
fig.tight_layout()

### $R_{Rup}$ Bias & (Residual) Standard Deviation

In [ ]:
res_dist_bias = res_dist_groups[ims].mean()
res_dist_std = res_dist_groups[ims].std()

colors = list(reversed(cm.viridis(np.linspace(0, 1, len(dist_bin_keys))))) 

fig, (ax1, ax2)  = plt.subplots(1, 2, figsize=(16, 6))

ax1.set_xlim(0.01, 10)
ax1.set_ylim(-1.5, 1.5)
ax1.grid(which="both", linewidth=0.5, alpha=0.5, linestyle="--")
ax1.axhline(0, color="black")

ax2.set_xlim(0.01, 10)
ax2.set_ylim(0, 1.5)
ax2.grid(which="both", linewidth=0.5, alpha=0.5, linestyle="--")

# Bias
ax1.semilogx(sr.constants.PERIODS, mar_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"], c="b", label="Overall")
for i, (cur_key, cur_group) in enumerate(res_dist_groups):
	ax1.semilogx(sr.constants.PERIODS, res_dist_bias.loc[cur_key, sr.constants.PSA_KEYS], label=f"{cur_key} - N = {res_dist_groups.size()[cur_key]}", c=colors[i])
ax1.legend(title="Distance Bin")


# Residual Standard Deviation
ax2.semilogx(sr.constants.PERIODS, mar_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], c="b")
for i, (cur_key, cur_group) in enumerate(res_dist_groups):
	ax2.semilogx(sr.constants.PERIODS, res_dist_std.loc[cur_key, sr.constants.PSA_KEYS], label=cur_key, c=colors[i])	

fig.tight_layout()

### Distance - Residual Scatter Plot

In [ ]:
plot_ims = ["pSA_0.01", "pSA_0.05", "pSA_0.1", "pSA_0.5", "pSA_1.0", "pSA_3.0", "pSA_5.0", "pSA_10.0"]
n_points_per_bin = 1000

fig, axs = mlt.plotting.get_fig_axes(8, 2, -1, (8, 6))

for cur_ax, cur_im in zip(axs, plot_ims):
    cur_ax.scatter(mar_res_df.rrup, mar_res_df[cur_im], alpha=0.5, s=5)

    bin_centres, bin_means, bin_stds = mlt.utils.compute_count_binned_trend(mar_res_df.rrup.values, mar_res_df[cur_im].values, n_points_per_bin, ignore_nans=True, verbose=False)
    cur_ax.plot(bin_centres, bin_means, c="r")
    cur_ax.plot(bin_centres, bin_means - bin_stds, linestyle="--", color="r", linewidth=2)
    cur_ax.plot(bin_centres, bin_means + bin_stds, linestyle="--", color="r", linewidth=2)

    cur_ax.set_title(sr.utils.get_nice_im_name(cur_im))
    cur_ax.set_xlabel("$R_{Rup}$")
    cur_ax.set_ylabel("Residual")
    cur_ax.set_xscale("log")
    cur_ax.set_ylim(-4, 4)
    cur_ax.grid(which="both", linewidth=0.5, alpha=0.5, linestyle="--")
    cur_ax.axhline(0, color="black")

fig.tight_layout()

## Combined

In [ ]:
plot_ims = ["pSA_0.01", "pSA_0.05", "pSA_0.1", "pSA_0.5", "pSA_1.0", "pSA_5.0", "pSA_10.0"]
grid_size = 40

n_cmin, n_cmax = 1, 2000
res_cmin, res_cmax = -2, 2

n_ims = len(plot_ims)
fig, axs = mlt.plotting.get_fig_axes(n_ims * 2, 2, -1, (8, 6))

for cur_im in plot_ims:
    ax1, ax2 = axs.pop(0), axs.pop(0)

    hb1 = ax1.hexbin(mar_res_df.rrup, mar_res_df.mag, gridsize=grid_size, cmap="Blues", bins="log", vmin=n_cmin, vmax=n_cmax)
    ax1.set_xlim(0, 200)
    ax1.set_ylim(3.5, 7.75)
    ax1.set_title(f"{sr.utils.get_nice_im_name(cur_im)} - Count")
    fig.colorbar(hb1, ax=ax1, label="log10(N)", pad=0)


    hb2 = ax2.hexbin(mar_res_df.rrup, mar_res_df.mag, C=mar_res_df[cur_im], gridsize=grid_size, cmap="coolwarm", vmin=res_cmin, vmax=res_cmax)
    ax2.set_xlim(0, 200)
    ax2.set_ylim(3.5, 7.75)
    ax2.set_title(f"{sr.utils.get_nice_im_name(cur_im)} - Residual")
    fig.colorbar(hb2, ax=ax2, label="Residual", pad=0)

fig.tight_layout()